In [ ]:
!pip install timm -q

# Vision Transformers (ViT)

Transformers took over NLP in 2017 (BERT, GPT). In 2020, Dosovitskiy et al. asked:  
**"What if we apply a pure Transformer to image patches — no convolutions at all?"**  
The answer was ViT — and it set a new state-of-the-art on ImageNet.

**By the end of this notebook you will be able to:**
- Explain how self-attention works on image patches
- Load and inspect a ViT model from TorchVision and `timm`
- Visualise attention maps to see what the model focuses on
- Fine-tune ViT on CIFAR10


## 1. From Pixels to Patches

A ViT treats an image as a **sequence of patches**, just like a sentence is a sequence of words.

```
224×224 image
  ↓ split into 16×16 patches
196 patches  (= (224/16)² )
  ↓ flatten each patch: 16×16×3 = 768 values
  ↓ linear projection → 768-dim embedding per patch
  ↓ prepend [CLS] token
197 tokens fed into Transformer encoder
  ↓
[CLS] output → classification head
```

**Key components:**
| Component | Role |
|-----------|------|
| Patch embedding | Splits image into patches, projects to embedding dim |
| Position encoding | Tells the model the spatial order of patches |
| [CLS] token | A learnable token whose final state is used for classification |
| Multi-Head Self-Attention | Each patch attends to all other patches |
| MLP head | Maps [CLS] output to class probabilities |


## 2. Self-Attention: How Patches Talk to Each Other

For each token (patch), self-attention computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- **Q** (Query): "What am I looking for?"
- **K** (Key): "What do I contain?"
- **V** (Value): "What information do I pass along?"

The softmax produces an **attention weight** for every pair of patches.  
A high weight between patch A and patch B means the model thinks they are relevant to each other.


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ViT_B_16_Weights
import timm
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Load ViT-B/16 from TorchVision
vit = models.vit_b_16(weights=ViT_B_16_Weights.DEFAULT).to(device)
vit.eval()

total  = sum(p.numel() for p in vit.parameters())
print(f"\nViT-B/16 Architecture:")
print(f"  Parameters  : {total/1e6:.1f}M")
print(f"  Image size  : 224×224")
print(f"  Patch size  : 16×16")
print(f"  Num patches : {(224//16)**2} + 1 CLS = 197 tokens")
print(f"  Embedding dim: 768")
print(f"  Layers      : 12 Transformer blocks")
print(f"  Heads       : 12 attention heads")


In [ ]:
# Compare ViT vs CNN families using timm
import timm

comparisons = {
    'AlexNet':         ('alexnet',              60.9,  56.5),
    'ResNet-50':       ('resnet50',             25.6,  80.9),
    'EfficientNet-B4': ('efficientnet_b4',      19.3,  83.4),
    'ConvNeXt-Tiny':   ('convnext_tiny',        28.6,  82.1),
    'ViT-B/16':        ('vit_base_patch16_224', 86.6,  81.1),
    'ViT-L/16':        ('vit_large_patch16_224',307.0, 82.6),
    'Swin-T':          ('swin_tiny_patch4_window7_224', 28.3, 81.4),
}

print(f"{'Model':<20} {'Params (M)':>12} {'ImageNet Top-1':>15} {'Type':<15}")
print("-"*65)
for name, (_, params, acc) in comparisons.items():
    arch = 'Transformer' if 'ViT' in name or 'Swin' in name else 'CNN'
    marker = '★' if acc == max(v[2] for v in comparisons.values()) else ' '
    print(f"{name:<20} {params:>12.1f} {acc:>14.1f}% {arch:<15} {marker}")

print("\n★ = best accuracy in this list")
print("\nKey insight: ViT needs MORE data & compute but scales better than CNNs.")
print("ConvNeXt/Swin show that CNN ideas + Transformer tricks = competitive performance.")


## 3. Attention Map Visualisation

We can inspect the attention weights of the [CLS] token in the last Transformer block.  
High attention weights on a spatial region = the model focuses there for classification.

This is ViT's equivalent of Grad-CAM — but built-in, not a post-hoc method.


In [ ]:
import torchvision.transforms as transforms
import torchvision.datasets as datasets

# Load one CIFAR10 image and upscale to 224×224 for ViT
tf_vit = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
ds = datasets.CIFAR10(root='./data', train=False, download=True, transform=tf_vit)
class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

# Hook to capture attention weights from the last block
attn_weights = {}
def hook_fn(module, input, output):
    attn_weights['last'] = output[1]  # (batch, heads, seq, seq)

# Register hook on the last encoder block's attention
vit.encoder.layers[-1].self_attention.register_forward_hook(hook_fn)

# Run 4 images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('ViT Attention Maps — [CLS] token attending to image patches', fontsize=13, fontweight='bold')

for col in range(4):
    img_tensor, label = ds[col * 100]
    with torch.no_grad():
        out = vit(img_tensor.unsqueeze(0).to(device))
    pred = out.argmax(1).item()

    # Mean attention from [CLS] (token 0) to all patches, averaged over heads
    attn = attn_weights['last'][0].mean(0)  # (seq, seq)
    cls_attn = attn[0, 1:]                  # [CLS] → patches (196 patches)
    side = int(cls_attn.shape[0]**0.5)
    cls_attn = cls_attn.reshape(side, side).cpu().numpy()
    cls_attn = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min())

    # Original image (denormalise)
    raw = img_tensor.permute(1,2,0).numpy()
    raw = raw * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
    raw = np.clip(raw, 0, 1)

    axes[0, col].imshow(raw)
    axes[0, col].set_title(f'True: {class_names[label]}\nPred: {class_names[pred]}',
                            color='green' if label==pred else 'red', fontsize=9)
    axes[0, col].axis('off')

    import torch.nn.functional as F
    attn_up = F.interpolate(torch.tensor(cls_attn).unsqueeze(0).unsqueeze(0),
                            size=(224,224), mode='bilinear', align_corners=False).squeeze().numpy()
    axes[1, col].imshow(raw)
    axes[1, col].imshow(attn_up, cmap='hot', alpha=0.5)
    axes[1, col].set_title('Attention Heatmap', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout(); plt.show()


## 4. Fine-Tuning ViT on CIFAR10

We use ViT as a feature extractor — freeze the backbone, train only the classification head.  
This is fast (only a small head trains) and works well even on a small dataset like CIFAR10.


In [ ]:
from tqdm.auto import tqdm

# ViT expects 224×224 — upscale CIFAR10
tf_train = transforms.Compose([
    transforms.Resize(224), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
tf_val = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
from torch.utils.data import DataLoader
train_ds = datasets.CIFAR10('./data', train=True,  download=True, transform=tf_train)
val_ds   = datasets.CIFAR10('./data', train=False, download=True, transform=tf_val)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

# Freeze backbone, replace head
vit_ft = models.vit_b_16(weights=ViT_B_16_Weights.DEFAULT).to(device)
for p in vit_ft.parameters(): p.requires_grad = False
vit_ft.heads = nn.Linear(768, 10).to(device)   # replace 1000-class head

trainable = sum(p.numel() for p in vit_ft.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,} (head only)")

optimizer = torch.optim.Adam(vit_ft.heads.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type=='cuda'):
                out  = model(imgs)
                loss = criterion(out, labels)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item(); correct += (out.argmax(1)==labels).sum().item(); total += len(labels)
    return total_loss/len(loader), correct/total

history = {'loss':[], 'acc':[]}
for epoch in range(5):
    tr_loss, _ = run_epoch(vit_ft, train_loader, train=True)
    vl_loss, vl_acc = run_epoch(vit_ft, val_loader, train=False)
    history['loss'].append(vl_loss); history['acc'].append(vl_acc)
    print(f"Epoch {epoch+1} | val_loss={vl_loss:.4f} | val_acc={vl_acc:.4f}")

fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].plot(history['loss']); axes[0].set(title='Val Loss', xlabel='Epoch'); axes[0].grid(alpha=0.3)
axes[1].plot(history['acc']);  axes[1].set(title='Val Accuracy', xlabel='Epoch'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"\nFinal val accuracy: {history['acc'][-1]:.4f}")
print("ViT achieves high accuracy with just 5 epochs of head-only training!")


## 5. Exercises

1. **Patch size experiment**: `timm.create_model('vit_base_patch32_224', pretrained=True, num_classes=10)` uses 32×32 patches (49 tokens vs 196). Compare accuracy and speed to ViT-B/16.
2. **Full fine-tuning**: Unfreeze all layers (`for p in vit_ft.parameters(): p.requires_grad = True`) and train with a smaller LR (1e-5). Does it improve over feature extraction?
3. **Attention heads**: Instead of averaging attention over all heads, plot each head separately. Do different heads attend to different parts of the image?
